# Exploratory motif analysis — full dataset
Corpus-scale analysis of the extracted `step_skip_leap` motif labels over the FULL
training corpus (full Irishman ~216k tunes + OpenScore Lieder), using the stored
`%motif:v1:` lines produced by the shared backbone (`motif/extract.py`, current logic,
Jun-30+ rebuild). Outputs: rank–frequency table/plots, corpus comparison, pattern-shape
statistics; summary CSV + figures saved under `../experiments/`.

In [ ]:
import os, sys, json, math
from collections import Counter, defaultdict
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
sys.path.insert(0, os.path.abspath('..'))

CORPORA = {
    'irish_full': '/usr/xtmp/cy232/accompgen/irishfull/abcfiles_processed_v1/C',
    'lieder':     os.path.abspath('../data/abcfiles_processed_v1/C'),
}
OUT_DIR = os.path.abspath('../experiments')
FIG_DIR = os.path.join(OUT_DIR, 'figures'); os.makedirs(FIG_DIR, exist_ok=True)


In [ ]:
# Read the stored %motif:v1 label of every piece (reference key C).
def read_labels(cdir):
    labels = []
    for fname in os.listdir(cdir):
        if not fname.endswith('_C.abc'):
            continue
        try:
            with open(os.path.join(cdir, fname), encoding='utf-8') as fh:
                for ln in fh:
                    if ln.startswith('%motif:v1:step_skip_leap:'):
                        pat = ln.split(':')[-1].strip()
                        if pat: labels.append(pat)
                        break
                    if ln.startswith('[V:'):
                        break
        except OSError:
            pass
    return labels

corpus_labels = {name: read_labels(d) for name, d in CORPORA.items()}
for name, ls in corpus_labels.items():
    print(f'{name}: {len(ls)} labelled pieces')
all_labels = [l for ls in corpus_labels.values() for l in ls]
counts = Counter(all_labels)
print(f'TOTAL: {len(all_labels)} pieces, {len(counts)} distinct motif patterns')

In [ ]:
# Rank-frequency table + cumulative coverage.
ranked = counts.most_common()
total = sum(counts.values())
cum = 0
rows = []
for rank, (pat, c) in enumerate(ranked, 1):
    cum += c
    rows.append({'rank': rank, 'motif': pat, 'count': c,
                 'share_pct': round(100*c/total, 3), 'cum_pct': round(100*cum/total, 2)})
import csv as _csv
with open(os.path.join(OUT_DIR, 'motif_distribution_full.csv'), 'w', newline='') as fh:
    w = _csv.DictWriter(fh, fieldnames=list(rows[0].keys())); w.writeheader(); w.writerows(rows)
print('top 20:')
for r in rows[:20]:
    print(f"  {r['rank']:>3d}  {r['motif']:14s} n={r['count']:<7d} {r['share_pct']:6.2f}%  cum {r['cum_pct']:5.1f}%")
for k in (5, 10, 20, 50, 100):
    if k <= len(rows):
        print(f'top-{k} coverage: {rows[k-1]["cum_pct"]}%')

In [ ]:
# Rank-frequency plot (log-log) + cumulative coverage.
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].loglog(range(1, len(ranked)+1), [c for _, c in ranked], marker='.', lw=0.8)
axes[0].set_xlabel('motif rank'); axes[0].set_ylabel('count'); axes[0].set_title('Motif rank-frequency (full corpus)')
axes[1].plot(range(1, len(rows)+1), [r['cum_pct'] for r in rows])
axes[1].set_xlabel('rank'); axes[1].set_ylabel('cumulative %'); axes[1].set_title('Cumulative coverage'); axes[1].set_ylim(0, 100)
plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, 'motif_rank_frequency_full.png'), dpi=200)
plt.show(); print('saved motif_rank_frequency_full.png')

In [ ]:
# Corpus comparison: Irish vs Lieder motif distributions.
irish_c = Counter(corpus_labels['irish_full']); lieder_c = Counter(corpus_labels['lieder'])
top_irish = [p for p,_ in irish_c.most_common(15)]
print(f"{'motif':14s} {'irish %':>8s} {'lieder %':>9s}")
for p in top_irish:
    print(f"{p:14s} {100*irish_c[p]/sum(irish_c.values()):8.2f} {100*lieder_c[p]/max(sum(lieder_c.values()),1):9.2f}")
overlap = set(p for p,_ in irish_c.most_common(20)) & set(p for p,_ in lieder_c.most_common(20))
print(f'\ntop-20 overlap between corpora: {len(overlap)}/20')

In [ ]:
# Pattern-shape statistics: length, direction balance, interval-class mix.
lens = Counter(); classes = Counter(); direction = Counter()
for pat, c in counts.items():
    xs = [int(t) for t in pat.split(',')]
    lens[len(xs)] += c
    for x in xs[1:]:
        classes[abs(x)] += c
        direction['up' if x > 0 else ('down' if x < 0 else 'flat')] += c
print('pattern length distribution:', dict(sorted(lens.items())))
tot_iv = sum(classes.values())
print('interval classes: ' + ', '.join(f'{k}({ {0:"unison",1:"step",2:"skip",3:"leap"}.get(k,k) })={100*v/tot_iv:.1f}%' for k,v in sorted(classes.items())))
tot_d = sum(direction.values())
print('direction: ' + ', '.join(f'{k}={100*v/tot_d:.1f}%' for k,v in direction.items()))

In [ ]:
# Where do the 11 evaluated motifs sit at full-corpus scale?
TESTED = ['0,-1,-1,-1','0,1,1,1','0,1,-1,-1','0,-1,-1,1','0,1,1,-1','0,3,-1,-1',
          '0,1,-1,1','0,-1,-2,2','0,2,-2,-1','0,3,3,3','0,-3,-3,-3']
rank_of = {r['motif']: r for r in rows}
print(f"{'motif':14s} {'rank':>6s} {'count':>8s} {'share%':>7s}")
for m in TESTED:
    r = rank_of.get(m)
    if r: print(f"{m:14s} {r['rank']:>6d} {r['count']:>8d} {r['share_pct']:>7.2f}")
    else: print(f"{m:14s} {'absent':>6s} {0:>8d} {0.0:>7.2f}")